# Health Data Cleaning & Preprocessing

A self-contained project by Jorgo Luka. Run the cells from top to bottom in Jupyter or Google Colab. All data is generated inside this notebook; no CSVs or Python files are required.

**Objective:** turn messy health observations into auditable, machine-learning-ready features. The data is synthetic, generated with seed 42, and describes no real people. This project demonstrates preprocessing, not medical prediction.

Python 3.10+ is recommended. If your environment lacks dependencies, run the optional installation cell below.

In [1]:
# Optional: uncomment in Jupyter/Colab if needed.
# %pip install "numpy>=1.26,<3" "pandas>=2.2,<3" "scikit-learn>=1.5,<2"

## 1. Imports and schema

Each ID represents one independent observation. Repeated observations from the same patient would require a patient-level split. The numeric bounds below are illustrative quality rules, not clinical standards.

In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = {'age': (18, 100), 'sleep_hours': (0, 24), 'steps': (0, 100000), 'resting_hr': (30, 220)}
REQUIRED = ['record_id', *NUMERIC, 'activity']
print('Expected columns:', REQUIRED)

Expected columns: ['record_id', 'age', 'sleep_hours', 'steps', 'resting_hr', 'activity']


## 2. Generate deliberately messy observations

The sample includes exact duplicates, numeric text, missing values, impossible measurements and inconsistent category formatting. Units: age in years, sleep in hours, steps per day and resting heart rate in beats per minute.

In [3]:
def sample_data(n=200, seed=42):
    """Artificial observations only; injected errors are intentional."""
    rng = np.random.default_rng(seed)
    frame = pd.DataFrame({'record_id': [f'S{i:04d}' for i in range(n)],
        'age': rng.integers(18, 85, n).astype(str),
        'sleep_hours': rng.uniform(4, 10, n).round(1).astype(str),
        'steps': rng.integers(500, 18000, n).astype(str),
        'resting_hr': rng.integers(45, 100, n).astype(str),
        'activity': rng.choice(['low', 'moderate', 'high', ' HIGH ', 'unknown'], n)})
    frame.loc[0, 'age'] = 'two'
    frame.loc[1, 'sleep_hours'] = '29'
    frame.loc[2, 'steps'] = '-30'
    frame.loc[3, 'resting_hr'] = ''
    frame.loc[4, 'age'] = '999'
    return pd.concat([frame, frame.iloc[:5]], ignore_index=True)

raw = sample_data()
print("Raw shape:", raw.shape)
print(raw.head(8).to_string(index=False))

Raw shape: (205, 6)
record_id age sleep_hours steps resting_hr activity
    S0000 two         9.5 11284         45  unknown
    S0001  69          29 14132         61    HIGH 
    S0002  61         5.6   -30         73     high
    S0003  47         9.8  2854             unknown
    S0004 999         8.7 14955         85      low
    S0005  75         8.3  9881         87  unknown
    S0006  23         6.7  6153         74  unknown
    S0007  64         5.6  9498         51     high


## 3. Inspect before cleaning

Numeric columns arrive as text. Empty strings and semantic errors need more than a simple `isna()` check.

In [4]:
print(raw.dtypes.to_string())
print("Exact duplicate rows:", int(raw.duplicated().sum()))
print("Raw activity values:")
print(raw.activity.value_counts(dropna=False).to_string())

record_id      object
age            object
sleep_hours    object
steps          object
resting_hr     object
activity       object
Exact duplicate rows: 5
Raw activity values:
activity
unknown     44
high        44
 HIGH       41
low         39
moderate    37


## 4. Clean with explicit rules

Remove exact duplicates; reject conflicting IDs. Parse numeric strings, recording missing, unparseable and out-of-range values separately. Invalid numbers become missing rather than fabricated values. Normalise activity labels and mark unsupported labels as missing. All transformations here use fixed rules, not statistics learned from the dataset.

In [5]:
def clean_data(raw):
    missing = set(REQUIRED) - set(raw.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    frame = raw[REQUIRED].copy()
    frame['record_id'] = frame['record_id'].astype('string').str.strip()
    if frame['record_id'].isna().any() or frame['record_id'].eq('').any():
        raise ValueError('Every observation needs a nonempty record_id')
    original_count = len(frame)
    frame = frame.drop_duplicates()
    duplicate_count = original_count - len(frame)
    if frame['record_id'].duplicated().any():
        raise ValueError('Conflicting observations share a record_id; resolve upstream')
    report = {'input_rows': original_count, 'exact_duplicates_removed': duplicate_count,
              'output_rows': len(frame), 'numeric_issues': {}}
    for column, (lower, upper) in NUMERIC.items():
        text = frame[column].astype('string').str.strip()
        absent = text.isna() | text.isin(['', 'NA', 'N/A', 'unknown'])
        values = pd.to_numeric(text.mask(absent), errors='coerce')
        invalid = values.notna() & ~values.between(lower, upper)
        report['numeric_issues'][column] = {
            'missing': int(absent.sum()),
            'unparseable': int((values.isna() & ~absent).sum()),
            'out_of_range': int(invalid.sum())}
        frame[column] = values.mask(invalid).astype(float)
    activity = frame['activity'].astype('string').str.strip().str.lower()
    valid = activity.isin(['low', 'moderate', 'high'])
    report['unknown_activity_values'] = int((~valid).sum())
    frame['activity'] = activity.where(valid).astype(object).where(valid, np.nan)
    report['missing_after_cleaning'] = {c: int(frame[c].isna().sum()) for c in REQUIRED}
    return frame.reset_index(drop=True), report

cleaned, report = clean_data(raw)
print(cleaned.head(8).to_string(index=False))
print(json.dumps(report, indent=2))

record_id  age  sleep_hours   steps  resting_hr activity
    S0000  NaN          9.5 11284.0        45.0      NaN
    S0001 69.0          NaN 14132.0        61.0     high
    S0002 61.0          5.6     NaN        73.0     high
    S0003 47.0          9.8  2854.0         NaN      NaN
    S0004  NaN          8.7 14955.0        85.0      low
    S0005 75.0          8.3  9881.0        87.0      NaN
    S0006 23.0          6.7  6153.0        74.0      NaN
    S0007 64.0          5.6  9498.0        51.0     high
{
  "input_rows": 205,
  "exact_duplicates_removed": 5,
  "output_rows": 200,
  "numeric_issues": {
    "age": {
      "missing": 0,
      "unparseable": 1,
      "out_of_range": 1
    },
    "sleep_hours": {
      "missing": 0,
      "unparseable": 0,
      "out_of_range": 1
    },
    "steps": {
      "missing": 0,
      "unparseable": 0,
      "out_of_range": 1
    },
    "resting_hr": {
      "missing": 1,
      "unparseable": 0,
      "out_of_range": 0
    }
  },
  "unknown_act

## 5. Compare data quality

The table shows missing values after parsing and validation. This is an honest view of uncertainty: invalid measurements have been exposed, not magically recovered.

In [6]:
quality = pd.DataFrame(report['numeric_issues']).T
quality['missing_after_cleaning'] = cleaned[list(NUMERIC)].isna().sum()
print(quality.to_string())
print('\nClean numeric summaries:')
print(cleaned[list(NUMERIC)].describe().round(2).to_string())

             missing  unparseable  out_of_range  missing_after_cleaning
age                0            1             1                       2
sleep_hours        0            0             1                       1
steps              0            0             1                       1
resting_hr         1            0             0                       1

Clean numeric summaries:
          age  sleep_hours     steps  resting_hr
count  198.00       199.00    199.00      199.00
mean    51.77         6.91   9690.32       71.19
std     18.37         1.80   4892.67       16.16
min     18.00         4.10    585.00       45.00
25%     37.00         5.20   5681.50       57.00
50%     51.00         6.90   9870.00       73.00
75%     68.00         8.60  13856.00       84.00
max     84.00        10.00  17941.00       99.00


## 6. Split before learning preprocessing statistics

Median imputation, scaling and encoding must fit only on training rows. ID is retained for traceability but excluded from the feature matrix. There is no target column or predictive model in this project.

In [7]:
train, test = train_test_split(cleaned, test_size=0.2, random_state=42)
assert set(train.record_id).isdisjoint(test.record_id)
print(f'Training rows: {len(train)}; test rows: {len(test)}')

Training rows: 160; test rows: 40


## 7. Impute, scale and encode

Numeric missing values use training medians; columns with training missingness receive indicators. StandardScaler learns training means and standard deviations. Activity missingness becomes an explicit unknown category. Unseen test categories are accepted and encoded as zeros in the category block.

In [8]:
def make_preprocessor():
    return ColumnTransformer([
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True,
                                                      keep_empty_features=True)),
                              ('scale', StandardScaler())]), list(NUMERIC)),
        ('category', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='unknown')),
                               ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
         ['activity'])], remainder='drop')

processor = make_preprocessor()
train_values = processor.fit_transform(train)
test_values = processor.transform(test)
feature_names = processor.get_feature_names_out()
train_features = pd.DataFrame(train_values, columns=feature_names, index=train.record_id)
test_features = pd.DataFrame(test_values, columns=feature_names, index=test.record_id)
print('Training feature shape:', train_features.shape)
print('Test feature shape:', test_features.shape)
print(train_features.head().round(3).to_string())

Training feature shape: (160, 12)
Test feature shape: (40, 12)
           numeric__age  numeric__sleep_hours  numeric__steps  numeric__resting_hr  numeric__missingindicator_age  numeric__missingindicator_sleep_hours  numeric__missingindicator_steps  numeric__missingindicator_resting_hr  category__activity_high  category__activity_low  category__activity_moderate  category__activity_unknown
record_id                                                                                                                                                                                                                                                                                                                                
S0079             0.555                 0.923           1.502                0.561                         -0.113                                 -0.079                           -0.079                                -0.079                      0.0                     0.0     

## 8. Validate the results

Check duplicate removal, known corruptions, idempotency, ID conflicts, finite output and training-only statistics. Challenge the fitted processor with an unseen category and confirm that transforming new data does not change learned statistics.

In [9]:
assert report['exact_duplicates_removed'] == 5
assert len(cleaned) == 200
assert pd.isna(cleaned.loc[0, 'age'])
assert pd.isna(cleaned.loc[1, 'sleep_hours'])
assert pd.isna(cleaned.loc[2, 'steps'])
assert pd.isna(cleaned.loc[3, 'resting_hr'])
assert pd.isna(cleaned.loc[4, 'age'])
assert clean_data(cleaned)[0].equals(cleaned)
assert np.isfinite(train_values).all() and np.isfinite(test_values).all()
imputer = processor.named_transformers_['numeric'].named_steps['impute']
np.testing.assert_allclose(imputer.statistics_, train[list(NUMERIC)].median().to_numpy())
stats_before = imputer.statistics_.copy()
challenge = test.head().copy()
challenge['activity'] = 'unseen'
challenge['age'] = 100
assert np.isfinite(processor.transform(challenge)).all()
np.testing.assert_array_equal(imputer.statistics_, stats_before)
conflicting = raw.iloc[:10].copy()
conflicting.loc[1, 'record_id'] = conflicting.loc[0, 'record_id']
try:
    clean_data(conflicting)
except ValueError as error:
    assert 'Conflicting' in str(error)
else:
    raise AssertionError('Conflicting IDs must be rejected')
print('All validation checks passed.')

All validation checks passed.


## Findings and limitations

The seeded sample has 205 rows and five exact duplicates, leaving 200 unique observations. Cleaning exposes two invalid ages, one invalid sleep duration, one negative step count and one missing resting heart rate. Unsupported activity values remain visible in the quality report. The split produces 160 training observations and 40 test observations.

The feature matrices are ready for a later model, but synthetic data cannot establish real-world health performance. Imputation can conceal informative missingness; indicators help retain some of that information. Fixed rules must be reviewed for a real dataset, including units, population, provenance and repeated patient records.

**Next step:** select a properly sourced dataset and define a prediction target. Exclude that target and future information from features before training any neural network. All results are embedded in this notebook; no additional project files are needed.